# ExoJump cloud model benchmark

Reference Google Colab workflow for the leakage-aware classical, CNN, BiLSTM, TCN, MiniROCKET, and modality-ablation experiments. The notebook operates on the anonymised event-window schema used by the reported benchmarks.

## 1. Install the public project

The maintained package and experiment scripts are installed directly from the public repository.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

PROJECT = Path('/content/exoskeleton-jump-recognition')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '-q', 'https://github.com/FelixZ01/exoskeleton-jump-recognition.git', str(PROJECT)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e',
    f'{PROJECT}[analysis,deep-learning,time-series]'
], check=True)
print('Project ready:', PROJECT)

## 2. Configure the anonymised benchmark dataset

The participant-level recording archive is not distributed publicly. This workflow expects an authorised, de-identified `jump_event_windows.npz` that follows the schema documented in `docs/DATA_DICTIONARY.md`.

In [ ]:
DATASET = Path('/content/jump_event_windows.npz')
OUTPUT_ROOT = Path('/content/cloud_results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert DATASET.exists(), f'Anonymised benchmark dataset not found at {DATASET}'
print('Dataset:', DATASET)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
import numpy as np, torch
arrays = np.load(DATASET, allow_pickle=False)
print({key: arrays[key].shape for key in arrays.files})
print('Participants:', sorted(np.unique(arrays['subject'].astype(str))))
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 3. Select the experiment profile

`QUICK_MODE=True` provides a short pipeline verification run. `QUICK_MODE=False` reproduces the formal three-seed configuration.

In [ ]:
QUICK_MODE = True
SEEDS = '42' if QUICK_MODE else '42,7,123'
EPOCHS = '3' if QUICK_MODE else '30'
PATIENCE = '2' if QUICK_MODE else '5'
CLASSICAL_MODELS = 'logistic,svm' if QUICK_MODE else 'logistic,svm,random_forest,minirocket'
print({'quick': QUICK_MODE, 'seeds': SEEDS, 'epochs': EPOCHS, 'device': DEVICE})

## 4. Classical and MiniROCKET baselines

In [ ]:
classical_output = OUTPUT_ROOT / 'classical_benchmark.json'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_classical_models.py'),
    '--dataset', str(DATASET), '--output', str(classical_output),
    '--models', CLASSICAL_MODELS, '--modalities', 'fusion',
    '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 5. CNN, BiLSTM, and TCN comparison

In [ ]:
architecture_output = OUTPUT_ROOT / 'deep_architectures'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_deep_models.py'),
    '--dataset', str(DATASET), '--output', str(architecture_output),
    '--architectures', 'cnn,bilstm,tcn', '--modalities', 'fusion',
    '--seeds', SEEDS, '--epochs', EPOCHS, '--patience', PATIENCE,
    '--device', DEVICE, '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 6. CNN modality ablation

The fusion CNN was trained in the previous cell, so this cell adds only IMU-only and sEMG-only controls.

In [ ]:
ablation_output = OUTPUT_ROOT / 'cnn_modality_ablation'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_deep_models.py'),
    '--dataset', str(DATASET), '--output', str(ablation_output),
    '--architectures', 'cnn', '--modalities', 'imu,semg',
    '--seeds', SEEDS, '--epochs', EPOCHS, '--patience', PATIENCE,
    '--device', DEVICE, '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 7. Produce the report table

In [ ]:
summary_path = OUTPUT_ROOT / 'MODEL_COMPARISON.md'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/summarize_benchmarks.py'),
    '--classical', str(classical_output),
    '--deep', str(architecture_output / 'deep_benchmark.json'),
    '--output', str(summary_path)
], cwd=PROJECT, check=True)
print(summary_path.read_text())
archive_base = OUTPUT_ROOT.parent / 'exojump_benchmark_results'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Results archive:', archive_path)